# Canonical end-to-end steering
Run setup on a T4 GPU, upload the bundle produced by `python -m sp_lense.research2.canonical_package release/canonical-pipeline.zip`, then execute. Jev API calls run locally; no credential or LoRA checkpoint is uploaded. The 64 cases were previously examined; this is a pipeline rerun, not an independent test.


In [ ]:
import torch,sys,subprocess,json
print(json.dumps({'gpu':torch.cuda.get_device_name(0),'torch':torch.__version__}))
subprocess.check_call([sys.executable,'-m','pip','install','-q','transformers==5.15.1'])


In [ ]:
import hashlib,json,os,shutil,subprocess,sys,time,zipfile
from pathlib import Path
from IPython.display import clear_output
bundle=Path('/content/canonical-pipeline.zip')
assert hashlib.sha256(bundle.read_bytes()).hexdigest()=='c40d1c4423d6d428e9b5966d5271f1103e67dba63e3e6bcb6be700ba7230ad63'
root=Path('/content/canonical')
root.mkdir(exist_ok=False)
with zipfile.ZipFile(bundle) as archive:
    assert all((root/n).resolve().is_relative_to(root.resolve()) for n in archive.namelist())
    archive.extractall(root)
out=root/'work/run'
env=os.environ.copy()
env.update(SP_LENSE_REPO=str(root),PYTHONPATH=str(root/'src'))
log=(root/'execution.log').open('w')
proc=subprocess.Popen([sys.executable,'-m','sp_lense.research2.canonical_pipeline',str(root),str(out)],env=env,stdout=log,stderr=subprocess.STDOUT)
began=time.monotonic()
try:
    while proc.poll() is None:
        if time.monotonic()-began>1800:
            proc.kill()
            proc.wait()
            raise TimeoutError('GPU time cap reached')
        clear_output(wait=True)
        print('Canonical detector -> base -> conditional controller -> guards')
        status=out/'STATUS.json'
        print(status.read_text() if status.exists() else 'Starting')
        time.sleep(5)
    print('Exit:',proc.returncode)
    for name in ['METRICS.json','FAILURE.json']:
        if (out/name).exists(): print(name,(out/name).read_text())
finally:
    if proc.poll() is None:
        proc.kill()
        proc.wait()
    log.close()
    out.mkdir(parents=True,exist_ok=True)
    shutil.copy2(root/'execution.log',out/'execution.log')
    shutil.make_archive('/content/canonical-pipeline-results','zip',out)
    print((root/'execution.log').read_text()[-3000:])


In [ ]:
from google.colab import files
print(hashlib.sha256(Path('/content/canonical-pipeline-results.zip').read_bytes()).hexdigest())
files.download('/content/canonical-pipeline-results.zip')
